In [2]:
import torch
import torch.nn.functional as F


In [14]:
torch.arange(4, dtype=torch.long).view(1,-1,1).expand(1,-1,4)

tensor([[[0, 0, 0, 0],
         [1, 1, 1, 1],
         [2, 2, 2, 2],
         [3, 3, 3, 3]]])

In [58]:
# write a same-idx-for-rows version first
# [current] = [refresh|denoising
# @torch.compile()
def concat_and_replace(matrix_origin, matrix_current, idx_current, shape_target): # (B, Hd, L, H)

    if matrix_origin.shape[-2] < shape_target[-2]:   # need patch
        length_patch = shape_target[-2] - matrix_origin.shape[-2]

        assert matrix_current.shape[-2] >= length_patch,\
            f'current shape should be >= patch shape, {matrix_current.shape[-2]} >= {length_patch}'
        matrix_patch = matrix_current[:, :, -length_patch:, :]   # TODO: check this

        matrix_origin = torch.cat([matrix_origin, matrix_patch], dim=-2)
    # end

    assert matrix_origin.shape[-2] == shape_target[-2],\
        f'origin shape should equal to target shape after patch, {matrix_origin.shape[-2]} == {shape_target[-2]}'

    # return matrix_origin.scatter(2, idx_current.view(1,1,-1,1).expand(1,32,-1,matrix_current.shape[-1]), matrix_current)
    matrix_origin[:, :, idx_current, :] = matrix_current
    return matrix_origin
# end

In [ ]:

# (BS, ?, L, K)
def concat_and_replace(self, matrix_origin, matrix_current, idx_current, shape_target):    # idx_current is 1d
    
    shape_target = [v_t if v_t != -1 else v_o for v_o, v_t in zip(matrix_origin.shape, shape_target)]

    device_origin = matrix_origin.device
    dtype_origin = matrix_origin.dtype
    shape_origin = matrix_origin.shape

    shape_current = list(matrix_origin.shape)
    shape_current[-2] = -1

    shape_mask = torch.ones(len(shape_target), dtype=torch.long, device=device_origin).tolist()
    shape_mask[-2] = -1

    if matrix_origin.shape[-2] < shape_target[-2]: # 扩展
        matrix_target = torch.zeros(shape_target, dtype=dtype_origin, device=device_origin)
        index_origin = torch.arange(matrix_origin.shape[-2], dtype=torch.long, device=device_origin).view(shape_mask).expand(shape_origin)
        print(matrix_target.shape, index_origin.shape, matrix_origin.shape)
        matrix_target.scatter_(-2, index_origin, matrix_origin)
    else:
        matrix_target = matrix_origin
    # end

    index_current = idx_current.view(shape_mask).expand(shape_current)
    matrix_target.scatter_(-2, index_current, matrix_current)

    return matrix_target
# end

In [74]:
matrix_origin = torch.tensor([[0,1,2],[3,4,5]], dtype=torch.long).view(1,2,-1,1).expand(1,2,-1,4)
matrix_current = torch.tensor([[7,8],[9,10]],dtype=torch.long).view(1,2,-1,1).expand(1,2,-1,4)
matrix_current2 = torch.tensor([[11,12],[13,14]],dtype=torch.long).view(1,2,-1,1).expand(1,2,-1,4)
idx_current = torch.tensor([[3,4]], dtype=torch.long)
idx_current2 = torch.tensor([[1,4]], dtype=torch.long)
len_target = 5

shape_target = list(matrix_origin.shape)
shape_target[-2] = len_target
shape_target[-1] = -1

a = concat_and_replace(matrix_origin, matrix_current, idx_current.squeeze(0), shape_target)
print(concat_and_replace(a, matrix_current2, idx_current2.squeeze(0), shape_target))

b = forge_and_replace(matrix_origin, matrix_current, idx_current.squeeze(0), shape_target)
print(forge_and_replace(b, matrix_current2, idx_current2.squeeze(0), shape_target))

tensor([[[[ 0,  0,  0,  0],
          [11, 11, 11, 11],
          [ 2,  2,  2,  2],
          [ 7,  7,  7,  7],
          [12, 12, 12, 12]],

         [[ 3,  3,  3,  3],
          [13, 13, 13, 13],
          [ 5,  5,  5,  5],
          [ 9,  9,  9,  9],
          [14, 14, 14, 14]]]])
[1, 2, 5, 4]
[1, 1, -1, 1]
[1, 2, 5, 4]
[1, 1, -1, 1]
tensor([[[[ 0,  0,  0,  0],
          [11, 11, 11, 11],
          [ 2,  2,  2,  2],
          [ 7,  7,  7,  7],
          [12, 12, 12, 12]],

         [[ 3,  3,  3,  3],
          [13, 13, 13, 13],
          [ 5,  5,  5,  5],
          [ 9,  9,  9,  9],
          [14, 14, 14, 14]]]])


In [ ]:

# from future_idx_selector import FutureIDXSelector, FutureIdxSelectorModelLoader, RandomModel
# device = 'cuda:0'


# class RandomModel:
#     def __call__(self, x, *args, **kwargs):
#         x = x[:,:,-1]
#         return torch.rand(x.shape, device=x.device).unsqueeze(-1)
#     # end
# # end

# loader_model = FutureIdxSelectorModelLoader(device)
# model = loader_model.load('mlp_3.pt')
# model = model.to(torch.bfloat16)
# # model = RandomModel()

# selector = FutureIDXSelector(model)

# # attn = torch.rand(24, dtype=torch.float64, device=device).reshape(1,24)
# attn = torch.rand(24, dtype=torch.bfloat16, device=device).reshape(1,24) - 0.5

# # print(attn)

# with torch.no_grad():
#     result = selector.select_future_by_attn(attn)
#     print(result)
# # end


# def select_future_by_3(model, met, h):  # (1, Q, 3)
#     attn = met[:, :, -1]    # (1, Q)
#     index_avail = (attn >0).nonzero(as_tuple=True)[1].reshape(attn.shape[0], -1)    # (1, Q)
#     index_avail_3 = index_avail.view(1,-1,1).expand(1,-1,3)
#     met_avail = torch.gather(met, 1, index_avail_3)
#     scores = model(met_avail).squeeze(-1)
    
#     idx = scores.argsort(dim=-1)[:, :h]
#     return torch.gather(index_avail, 1, idx)
# # end

# met = torch.rand(24, dtype=torch.bfloat16, device=device).reshape(1,-1,3) - 0.5

# select_future_by_3(model, met, 5)